# Silver Layer: Posts Transformation

## Overview
This Silver layer transforms raw posts data into a clean, structured, and analytics-ready dataset.  
It standardizes formats, enriches attributes, and prepares the data for downstream consumption.

---

## Purpose
To clean and enrich raw posts data, ensuring consistency, usability, and reliability for analytics and reporting layers.

---

## Transformations Applied

### 1. Tags Normalization
- Converts the raw `tags` string into a structured array format (`TagsArray`)
- Improves usability for filtering and aggregation

### 2. Column Standardization
- Renames columns to follow a consistent naming convention
- Example: `Id` → `PostId`

### 3. Post Type Mapping
- Maps `PostTypeId` to human-readable labels
- Improves interpretability of post categories (e.g., Question, Answer)

---

## Output

- **Dataset:** `stg_posts_df`
- **Layer:** Silver
- **Format:** Spark DataFrame / Delta Table (if persisted)
- **Description:** Cleaned and enriched posts dataset ready for analytics and downstream processing

---

## Notes
- This layer does not perform business aggregations
- It ensures data quality and consistency before Gold layer transformations
- Designed to be idempotent and reusable in batch pipelines

In [0]:
raw_posts_df = spark.table("`data-plataform-jayzern`.default.posts")

In [0]:
display(raw_posts_df.limit(10))

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame

# Declarative functions to do transformations:

def normalize_tags(df: DataFrame) -> DataFrame:
    """
    Converts pipe-delimited tag string into array format.

    Input contract:
        - tags: string (pipe-delimited, e.g. 'python|spark|sql')

    Output contract:
        - TagsArray: array<string>
        - Original 'tags' column removed

    Notes:
        - Removes empty tag values
    """
    return (
        df.withColumn(
            "TagsArray",
            F.filter(F.split(F.col("tags"), r"\|"), lambda x: x != "")
        )
        .drop("tags")
    )

def standardize_post_schema(df: DataFrame) -> DataFrame:
    """
    Standardizes column naming.

    Input contract:
        - Id column exists

    Output contract:
        - Id renamed to PostId
    """
    return df.withColumnRenamed("Id", "PostId")

def map_post_type(df: DataFrame) -> DataFrame:
    """
    Enriches posts with human-readable PostType.

    Input contract:
        - PostTypeId: int (nullable = false)

    Output contract:
        - Adds column 'PostType': string
        - Preserves all existing rows (left join)

    Business context:
        Translates internal StackOverflow post type IDs into descriptive labels
        for downstream analytics.

    Implementation note:
        Uses broadcast join due to small static lookup table.
    """
    map_data = [
        (1, "Question"),
        (2, "Answer"),
        (3, "Orphaned tag wiki"),
        (4, "Tag wiki excerpt"),
        (5, "Tag wiki"),
        (6, "Moderator nomination"),
        (7, "Wiki placeholder"),
        (8, "Privilege wiki"),
        (9, "Article"),
        (10, "HelpArticle"),
        (12, "Collection"),
        (13, "ModeratorQuestionnaireResponse"),
        (14, "Announcement"),
        (15, "CollectiveDiscussion"),
        (17, "CollectiveCollection")
    ]

    map_df = spark.createDataFrame(map_data, ["PostTypeId", "PostType"])

    return df.join(F.broadcast(map_df), "PostTypeId", "left")

In [0]:
stg_posts_df = (
    raw_posts_df
    .transform(normalize_tags)
    .transform(standardize_post_schema)
    .transform(map_post_type)
)

In [0]:
display(stg_posts_df.limit(5))

In [0]:
stg_posts_df.printSchema()

In [0]:
def validate_stg_posts(df):
    """
    Validates the staging posts dataframe after transformations.

    Purpose:
        Ensures that key transformations were applied correctly and that
        the dataset is reliable for downstream analysis.

    Checks performed:
        1. PostType mapping completeness (no null PostType values)
        2. Tags transformation correctness (no empty tag arrays)
        3. Schema integrity (required columns exist)

    Output:
        - Prints warnings if issues are found
        - Returns a list of warning messages (empty if all valid)
    """
    warnings = []

    # Validate PostType mapping
    null_posttypes = df.filter(F.col("PostType").isNull()).count()
    if null_posttypes > 0:
        warnings.append(
            f"PostType validation failed: {null_posttypes} rows missing PostType"
        )

    # Validate TagsArray
    empty_tags = df.filter(F.size("TagsArray") == 0).count()
    if empty_tags > 0:
        warnings.append(
            f"TagsArray validation failed: {empty_tags} rows are empty"
        )

    # Validate schema
    required_columns = ["PostId", "TagsArray", "PostType"]
    missing_columns = [c for c in required_columns if c not in df.columns]

    if missing_columns:
        warnings.append(
            f"Schema validation failed: missing columns {missing_columns}"
        )

    # Output result
    if warnings:
        print("\n".join(warnings))
    else:
        print("All validations passed successfully")

    return warnings

warnings = validate_stg_posts(stg_posts_df)

In [0]:
display(stg_posts_df.limit(5))
stg_posts_df.printSchema()

# Delta Upsert Pipeline
~ How data is saved safely over time

## Overview
This function performs a safe and idempotent upsert into a Delta Lake table.

It ensures that incoming data is either:
- Inserted if it is new
- Updated if it already exists

## Design Principles
- No pre-filtering of data (avoids missing late-arriving updates)
- Uses Delta MERGE for correctness
- Fully idempotent (safe to rerun multiple times)
- Supports full refresh mode for rebuilding tables

## Inputs
- dest_table: Target Delta table name
- df: Incoming Spark DataFrame
- unique_key: Business key used for matching rows (e.g. PostId)
- full_refresh: If True, overwrites entire table

## Behavior
### Full Refresh Mode
If the table does not exist or full_refresh=True:
- Entire table is overwritten with incoming data

### Incremental Mode
If the table exists:
- Rows are matched using unique_key
- Existing rows are updated
- New rows are inserted

## Notes
This implementation prioritizes correctness over performance optimization.
It is suitable for batch ETL pipelines in Databricks using Delta Lake.

In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F
from pyspark.sql import DataFrame

def delta_upsert(
    dest_table: str,
    df: DataFrame,
    unique_key: str,
    full_refresh: bool = False
):
    """
    Performs a safe Delta Lake upsert (MERGE).

    Design:
        - No pre-filtering (avoids missing late updates)
        - Fully idempotent MERGE
        - Supports insert + update
        - Optional full refresh for rebuilds

    Parameters:
        dest_table (str): Target Delta table name
        df (DataFrame): Incoming dataset
        unique_key (str): Business key for matching rows (e.g., PostId)
        full_refresh (bool): If True, overwrites table completely

    Behavior:
        - If table doesn't exist OR full_refresh=True → overwrite
        - Else → MERGE (update existing + insert new)
    """

    # -----------------------------
    # CASE 1: Full refresh
    # -----------------------------
    if full_refresh or not spark.catalog.tableExists(dest_table):

        (
            df.write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(dest_table)
        )

        return

    # -----------------------------
    # CASE 2: Incremental MERGE
    # -----------------------------
    delta_table = DeltaTable.forName(spark, dest_table)

    (
        delta_table.alias("t")
        .merge(
            source=df.alias("s"),
            condition=f"t.{unique_key} = s.{unique_key}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

dest_table = "`data-plataform-jayzern`.default.stg_posts"

delta_upsert(
    dest_table=dest_table,
    df=stg_posts_df,
    unique_key="PostId",
    full_refresh=False
)

In [0]:
display(spark.table(dest_table).limit(5))